In [1]:
#%pip install -U langchain-ollama
#%pip install -U langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers


   -------------------- ------------------- 1/2 [langchain-ollama]
   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os

print(os.path.exists("tcs.pdf"))

True


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
# import faiss
import os
from langchain_community.vectorstores import FAISS
from openai import OpenAI

C:\Users\Asus\AppData\Local\Temp\ipykernel_16196\2602581205.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
#Step-2-Load tcs pdf file and load all the pages
loader = PyPDFLoader('tcs.pdf')
mydocuments = loader.load()


In [3]:
## step 3 - convert documents to chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=500
)

mychunks = text_splitter.split_documents(mydocuments)

print("Total chunks:", len(mychunks))
print(mychunks[0].page_content)



Total chunks: 640
Integrated
/A.caltnnual Report
2025-26
Leadership in 
Enterprise /A.caltI
Infrastructure 
to Intelligence


In [4]:
## step 4 - take vector model
model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
## step 5 - create faiss db and pass documents and vector model
vector_store = FAISS.from_documents(mychunks, model)

In [6]:
vector_store = FAISS.load_local(
    "tcs_faiss_index",
    model,
    allow_dangerous_deserialization=True
)

RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss\faiss\faiss\impl\io.cpp:70: Error: 'f' failed: could not open tcs_faiss_index\index.faiss for reading: No such file or directory

In [8]:
# Step 6: Retrieve the 3 most relevant chunks from FAISS

question = "How is TCS performing in terms of revenue?"

results = vector_store.similarity_search(
    question,
    k=3
)

# Step 7: Combine all 3 chunks into one variable

our_context = ""

for res in results:
    our_context += res.page_content + "\n\n"

print(our_context)

of margins.
These were offset by investments in talent and capability building, 
and other investments for strengthening ecosystem partnerships and 
increasing Go-to-Market pipeline.
Operating Profit Trend (H crore)
FY  2022
48, 453
FY  2024
59 , 31 1
FY  2023
54, 2 37
FY  2025
62 , 1 65
FY  2026
66, 838
2 5. 3% 2 5. 0 %
2 4. 3%2 4. 6%2 4. 1 %
O perati ng  P rof i t
O perati ng  Mar g i n
Over the past five fiscal years, TCS has steadily increased its Earnings 
Per Share (EPS), recording a compound annual growth rate of 10.3% . 
This consistent upward trend demonstrates not only the company’s 
strengthening profits but also its dedication to providing sustained value 
for shareholders.
Earnings per share (Amount in H)
10. 3%
C AG R
FY  2022
 1 0 3. 62  
FY  2024
 1 2 7. 74 
FY  2023
 1 1 5. 1 9  
FY  2025
 1 34. 1 9  
FY  2026
 1 45. 9 9  
TCS has demonstrated a strong cash conversion ratio, which reflects its solid 
financial position and effective generation of operational cash. 
A r

In [ ]:
## step 7 - keep all 3 chunks results into one variable
our_context = ''
for res in results:
	final_result += res.page_content+'\n\n'

In [9]:
## step 8 - make a prompt
our_prompt = f"""
		I am going to ask a question based on my contxt. please answer the
		question based on my context only. if you dont find answer in my context
		then tell me you did nt find the answer, and dont give me answer from
                your knowledge.

		context = {our_context}
		question = {question}
	"""

In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

response = llm.invoke(our_prompt)

print(response.content)

Based on the provided context, TCS is performing well in terms of revenue. According to the context, TCS has recorded a compound annual growth rate (CAGR) of 10.2% over the past five financial years, highlighting the resilience and scalability of its business model. In FY 2026, TCS achieved a year-over-year revenue growth of 4.6%, led by focused and effective execution. Additionally, the context mentions that TCS has consistently increased its Earnings Per Share (EPS) and has demonstrated a strong cash conversion ratio, indicating a solid financial position and effective generation of operational cash.
